# 🧠 Brain Tumor MRI Classifier
## Notebook d'entraînement complet

**Dataset** : [Kaggle Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)  
**Modèle** : EfficientNetB0 + Transfer Learning  
**Classes** : Gliome, Méningiome, Pas de tumeur, Tumeur pituitaire


## 0. Installation et imports

In [ ]:
# Installer les dépendances (Google Colab)
# !pip install -r requirements.txt

# Pour Kaggle : configurer l'API
# !pip install kaggle
# from google.colab import files
# files.upload()  # uploader kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.insert(0, 'src')

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU disponibles    : {tf.config.list_physical_devices("GPU")}')

## 1. Téléchargement du dataset Kaggle

In [ ]:
# Télécharger le dataset depuis Kaggle
# !kaggle datasets download masoudnickparvar/brain-tumor-mri-dataset
# !unzip brain-tumor-mri-dataset.zip -d data/

# Structure attendue :
# data/
#   Training/
#     glioma/       (1321 images)
#     meningioma/   (1339 images)
#     notumor/      (1595 images)
#     pituitary/    (1457 images)
#   Testing/
#     glioma/       (300 images)
#     meningioma/   (306 images)
#     notumor/      (405 images)
#     pituitary/    (300 images)

DATA_DIR = 'data'  # Modifier si nécessaire
print(f'Dataset dans : {DATA_DIR}')

## 2. Exploration des données

In [ ]:
from data_preprocessing import explore_dataset, show_sample_images

# Distribution des classes
explore_dataset(DATA_DIR, save_dir='outputs')

# Afficher la figure sauvegardée
from IPython.display import Image as IPImage
IPImage('outputs/distribution.png')

In [ ]:
# Exemples d'images par classe
show_sample_images(DATA_DIR, save_dir='outputs')
IPImage('outputs/samples.png')

## 3. Préparation des données

In [ ]:
from data_preprocessing import get_data_generators

train_gen, val_gen, test_gen, class_weights = get_data_generators(
    DATA_DIR, val_split=0.15
)

print('\nPoids des classes (pour le déséquilibre) :')
print(class_weights)

## 4. Construction du modèle

In [ ]:
from model import build_model, compile_model, print_model_summary

model = build_model()
model = compile_model(model, learning_rate=1e-3)
print_model_summary(model)

## 5. Entraînement — Phase 1 (tête de classification)

In [ ]:
from model import get_callbacks

callbacks_p1 = get_callbacks(
    checkpoint_path='outputs/models/phase1_best.h5',
    log_dir='outputs/logs/phase1',
    patience=8
)

history_p1 = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_p1,
    verbose=1
)

print(f"\nMeilleure val_accuracy phase 1 : {max(history_p1.history['val_accuracy']):.4f}")

## 6. Entraînement — Phase 2 (fine-tuning)

In [ ]:
from model import unfreeze_top_layers

# Dégeler les 20 dernières couches
unfreeze_top_layers(model, num_layers=20)
model = compile_model(model, learning_rate=1e-5)  # LR 100x plus faible

callbacks_p2 = get_callbacks(
    checkpoint_path='outputs/models/best_model.h5',
    log_dir='outputs/logs/phase2',
    patience=6
)

history_p2 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_p2,
    verbose=1
)

print(f"\nMeilleure val_accuracy phase 2 : {max(history_p2.history['val_accuracy']):.4f}")

## 7. Évaluation

In [ ]:
from evaluation import (
    get_predictions, plot_confusion_matrix,
    print_classification_report, plot_roc_curves
)

# Évaluation finale sur le jeu de test
test_results = model.evaluate(test_gen, verbose=1)
metrics_names = ['loss', 'accuracy', 'auc', 'precision', 'recall']
for name, val in zip(metrics_names, test_results):
    print(f'  {name:12s}: {val:.4f}')

In [ ]:
# Prédictions et matrice de confusion
y_true, y_pred, y_proba = get_predictions(model, test_gen)

fig = plot_confusion_matrix(y_true, y_pred, save_path='outputs/confusion_matrix.png')
plt.show()

report = print_classification_report(y_true, y_pred)

In [ ]:
# Courbes ROC
fig = plot_roc_curves(y_true, y_proba, save_path='outputs/roc_curves.png')
plt.show()

## 8. Grad-CAM — Visualisation de l'attention

In [ ]:
from evaluation import visualize_gradcam
from inference import predict
from PIL import Image
import os

# Charger une image de test
sample_path = 'data/Testing/glioma/'
sample_img  = os.listdir(sample_path)[0]
img = Image.open(os.path.join(sample_path, sample_img))

# Prédiction
result = predict(model, img)
print(f"Classe prédite : {result['predicted_label_fr']}")
print(f"Confiance      : {result['confidence']:.2%}")

# Visualisation Grad-CAM
fig, gradcam = visualize_gradcam(img, model, result['predicted_class'], true_class='glioma')
plt.show()

## 9. Sauvegarde du modèle

In [ ]:
import json

# Sauvegarder le modèle
model.save('outputs/models/brain_tumor_model.h5')
print('Modèle sauvegardé → outputs/models/brain_tumor_model.h5')

# Sauvegarder les infos
info = {
    'test_accuracy': float(test_results[1]),
    'test_auc': float(test_results[2]),
    'classes': ['glioma', 'meningioma', 'notumor', 'pituitary'],
    'img_size': [224, 224],
    'model_architecture': 'EfficientNetB0',
}
with open('outputs/models/model_info.json', 'w') as f:
    json.dump(info, f, indent=2)
print('Infos modèle    → outputs/models/model_info.json')

## 10. Lancement de l'application Streamlit

```bash
# Dans un terminal, depuis le dossier brain_tumor_app/
streamlit run app.py
```

Ou pour l'exposer publiquement depuis Colab :
```python
# !pip install pyngrok
# from pyngrok import ngrok
# !streamlit run app.py &
# public_url = ngrok.connect(8501)
# print(f'URL publique : {public_url}')
```